# Implementasi Simulasi Sistem Bengkel (TA-05)

**Nama:** Bayu Muhammad Ramadhan  
**NIM:** 301240008  
**Kelas:** 5A

## Deskripsi
Program memodelkan alur pelayanan bengkel dengan dua montir. Mobil datang dan mendaftar di resepsionis. Jika montir tersedia, mobil langsung diperbaiki; jika kedua montir sibuk, mobil masuk antrian FIFO. Setelah perbaikan selesai, pemilik membayar di kasir dan mobil keluar.

## Asumsi pemodelan
1. Terdapat dua montir.
2. Antrian menggunakan prinsip FIFO (First In, First Out).
3. Waktu pendaftaran dan pembayaran dianggap nol/seketika karena durasinya tidak ditentukan pada soal.
4. Waktu kedatangan dan waktu perbaikan pada contoh merupakan data uji dalam satuan menit.
5. Kejadian selesai perbaikan diproses lebih dahulu jika waktunya sama dengan kedatangan baru, sehingga montir yang baru selesai dapat langsung digunakan.

In [1]:
from collections import deque
import heapq

def simulasi_bengkel(waktu_kedatangan, waktu_perbaikan):
    """
    Simulasi DES bengkel dengan 2 montir dan antrian FIFO.
    Pendaftaran dan pembayaran dianggap berlangsung seketika karena
    durasinya tidak ditentukan dalam soal.
    """
    jumlah_montir = 2
    montir_bebas = [1, 2]
    antrian = deque()
    kalender = []
    urutan = 0
    catatan = []
    hasil = {}

    # Jadwalkan kedatangan setiap mobil.
    for mobil_id, waktu_datang in enumerate(waktu_kedatangan, start=1):
        urutan += 1
        # Prioritas 1 untuk ARRIVAL; DEPARTURE diproses lebih dahulu
        # bila waktunya sama agar montir yang baru selesai dapat digunakan.
        heapq.heappush(kalender, (waktu_datang, 1, urutan, "ARRIVAL", mobil_id))

    while kalender:
        waktu, _, _, jenis_event, mobil_id = heapq.heappop(kalender)

        if jenis_event == "ARRIVAL":
            catatan.append(
                f"{waktu:6.2f} | Mobil {mobil_id:02d} | Tiba dan mendaftar di resepsionis"
            )
            hasil[mobil_id] = {
                "kedatangan": waktu,
                "mulai_perbaikan": None,
                "selesai_perbaikan": None,
                "montir": None,
            }

            if montir_bebas:
                montir_id = montir_bebas.pop(0)
                hasil[mobil_id]["montir"] = montir_id
                hasil[mobil_id]["mulai_perbaikan"] = waktu
                selesai = waktu + waktu_perbaikan[mobil_id - 1]
                urutan += 1
                heapq.heappush(
                    kalender, (selesai, 0, urutan, "DEPARTURE", mobil_id)
                )
                catatan.append(
                    f"{waktu:6.2f} | Mobil {mobil_id:02d} | Montir {montir_id} mulai memperbaiki"
                )
            else:
                antrian.append(mobil_id)
                catatan.append(
                    f"{waktu:6.2f} | Mobil {mobil_id:02d} | Kedua montir sibuk; masuk antrian"
                )

        elif jenis_event == "DEPARTURE":
            montir_id = hasil[mobil_id]["montir"]
            hasil[mobil_id]["selesai_perbaikan"] = waktu
            catatan.append(
                f"{waktu:6.2f} | Mobil {mobil_id:02d} | Perbaikan selesai oleh montir {montir_id}"
            )
            catatan.append(
                f"{waktu:6.2f} | Mobil {mobil_id:02d} | Pemilik membayar di kasir; mobil keluar"
            )

            if antrian:
                mobil_berikut = antrian.popleft()
                hasil[mobil_berikut]["montir"] = montir_id
                hasil[mobil_berikut]["mulai_perbaikan"] = waktu
                selesai = waktu + waktu_perbaikan[mobil_berikut - 1]
                urutan += 1
                heapq.heappush(
                    kalender, (selesai, 0, urutan, "DEPARTURE", mobil_berikut)
                )
                catatan.append(
                    f"{waktu:6.2f} | Mobil {mobil_berikut:02d} | Keluar dari antrian; montir {montir_id} mulai memperbaiki"
                )
            else:
                montir_bebas.append(montir_id)
                montir_bebas.sort()

    print("=== SIMULASI SISTEM BENGKEL (2 MONTIR) ===")
    print("Waktu  | Aktivitas")
    print("-" * 78)
    for baris in catatan:
        print(baris)

    print("\n=== RINGKASAN ===")
    print(f"Jumlah mobil selesai dilayani : {len(hasil)}")
    total_tunggu = 0.0
    for mobil_id in sorted(hasil):
        data = hasil[mobil_id]
        tunggu = data["mulai_perbaikan"] - data["kedatangan"]
        total_tunggu += tunggu
        print(
            f"Mobil {mobil_id:02d}: montir {data['montir']}, "
            f"waktu tunggu {tunggu:.2f}, "
            f"selesai {data['selesai_perbaikan']:.2f}"
        )
    rata_tunggu = total_tunggu / len(hasil) if hasil else 0.0
    print(f"Rata-rata waktu tunggu          : {rata_tunggu:.2f}")
    print(f"Total waktu simulasi            : {max((x['selesai_perbaikan'] for x in hasil.values()), default=0):.2f}")

    return hasil, catatan


# Data contoh untuk pengujian (satuan waktu: menit).
waktu_kedatangan = [0, 1, 2, 3, 7, 8]
waktu_perbaikan = [5, 4, 6, 3, 2, 4]

hasil, log_simulasi = simulasi_bengkel(waktu_kedatangan, waktu_perbaikan)


=== SIMULASI SISTEM BENGKEL (2 MONTIR) ===
Waktu  | Aktivitas
------------------------------------------------------------------------------
  0.00 | Mobil 01 | Tiba dan mendaftar di resepsionis
  0.00 | Mobil 01 | Montir 1 mulai memperbaiki
  1.00 | Mobil 02 | Tiba dan mendaftar di resepsionis
  1.00 | Mobil 02 | Montir 2 mulai memperbaiki
  2.00 | Mobil 03 | Tiba dan mendaftar di resepsionis
  2.00 | Mobil 03 | Kedua montir sibuk; masuk antrian
  3.00 | Mobil 04 | Tiba dan mendaftar di resepsionis
  3.00 | Mobil 04 | Kedua montir sibuk; masuk antrian
  5.00 | Mobil 01 | Perbaikan selesai oleh montir 1
  5.00 | Mobil 01 | Pemilik membayar di kasir; mobil keluar
  5.00 | Mobil 03 | Keluar dari antrian; montir 1 mulai memperbaiki
  5.00 | Mobil 02 | Perbaikan selesai oleh montir 2
  5.00 | Mobil 02 | Pemilik membayar di kasir; mobil keluar
  5.00 | Mobil 04 | Keluar dari antrian; montir 2 mulai memperbaiki
  7.00 | Mobil 05 | Tiba dan mendaftar di resepsionis
  7.00 | Mobil 05 | Kedua m

## Penjelasan hasil uji
Dengan data contoh enam mobil, seluruh mobil selesai dilayani. Rata-rata waktu tunggu pada data uji adalah **1,33 menit**, dan waktu simulasi berakhir pada **14 menit**. Mobil yang datang ketika kedua montir sibuk menunggu sampai montir tersedia, lalu dilayani berdasarkan urutan kedatangan.

## Kesimpulan
Model telah merepresentasikan dua montir, antrian FIFO, proses kedatangan, perbaikan, pembayaran, dan keberangkatan. Durasi pendaftaran serta pembayaran dapat ditambahkan sebagai event tersendiri apabila data waktunya tersedia.